# Bird Species Identifier
A model for identifying the species of bird from audio.

## Imports

In [40]:
import pandas as pd 
import numpy as np 
import tensorflow as tf 
from pathlib import Path
import os
import sys
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D, concatenate, Conv1D, MaxPooling1D
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_curve, auc, mean_squared_error
import matplotlib.pyplot as plt 
from collections.abc import Sequence
from sklearn import preprocessing
%matplotlib inline
import csv
import glob
from IPython.display import Image
import seaborn as sns


## Global Control Flow Flags and Program Configuration

In [41]:
ITERATION = 0
PAUL = True # paul, you are running in an environment with a different keras backend than us, and are using pytorch instead of tensorflow.
# use this flag to gate code that should be run when only you want it to run. If this feels like a clunky and bad idea, feel free to disregard this. 
# In general, my idea for these flags was they could be used to section off highly experimental / broken code, or code that only works in a specific
# environment that others might not have. 
RYAN = True
BEN = True
WINDOW_SIZE = 7
OPTIMIZER_LEARNING_RATE = 0.001


## Define Helper Methods

In [42]:
def plot_losses(history, base_path, iteration:int):
    # Plot training & validation loss over epochs
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.ylim(bottom=0.0, top=10.0)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs. Validation Loss")
    plt.legend()
    plt.savefig(
        os.path.join(base_path, f"training-validiation-loss--epoch---Model {iteration}")
    )
    plt.close()


def print_schema(dataframe: pd.DataFrame):
    print('~~~~~~dataframe schema~~~~~~')
    print(f"Dataframe shape: {dataframe.shape} | Dataframe length: {len(dataframe)}")
    print('Column labels: ')
    print(dataframe.columns)
    print('Dataframe head: ')
    print(f"{dataframe.head()}")
def print_column(dataframe: pd.DataFrame, columns: str | list[str]):
    if isinstance(columns, list):
        for i, label in enumerate(columns):
            print(f"column {i}")
            print(dataframe[label])
    else:
        print(dataframe[columns])
# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column. 
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    #if target_type in (np.int64, np.int32):
        ## Classification
        #dummies = pd.get_dummies(df[target])
        #return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    #else#:
        ## Regression
    return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(path, pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    b = plt.plot(t['pred'].tolist(),label='prediction')
    a = plt.plot(t['y'].tolist(),label='expected')

    plt.ylabel('output')
    plt.legend()
    plt.savefig(path)
    plt.close()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low
# get all files in a directory. Use extension_filter to only grab files with that extension
# usage: pdf_list = get_all_files_(os.getcwd(), ".pdf")
def get_all_files(base_directory_path, extension_filter = None):
    file_list = []
    for directory in os.listdir(base_directory_path):
        subdirectory_path = os.path.join(base_directory_path, directory)
        for file_path in os.listdir(subdirectory_path):
            #print(file_path)
            if os.path.isfile(os.path.join(subdirectory_path, file_path)):
                print('hit')
                if extension_filter is None:
                    file_list.append(file_path)
                else:
                    _, file_extension = os.path.splitext(file_path)
                    if file_extension == extension_filter:
                        file_list.append(file_path)
    return file_list

## Configure Environment

In [43]:
base_path = os.path.join(os.getcwd(), "output")
iteration_path = os.path.join(base_path, f"iteration-{ITERATION}")
mp3_dataset_base_path = os.path.join(os.getcwd(), "data/charaNet")
spectrogram_dataset_base_path = os.path.join(os.getcwd(), 'data/spectrogram_dataset')
spectrogram_train_path = os.path.join(spectrogram_dataset_base_path, 'train')
spectrogram_test_path = os.path.join(spectrogram_dataset_base_path, 'test')
spectrogram_val_path = os.path.join(spectrogram_dataset_base_path, 'val')
try:
    os.mkdir(spectrogram_dataset_base_path)
    os.mkdir(spectrogram_train_path)
    os.mkdir(spectrogram_test_path)
    os.mkdir(spectrogram_val_path)
except FileExistsError as e:
    print('That Dataset Folder already exists')
    print(e)

try:
    os.mkdir(base_path)
except FileExistsError as e:
    print(f"{base_path} already exists")
except OSError as e:
    print(f"Error creating directory: {base_path}")
try:
    os.mkdir(iteration_path)
except FileExistsError as e:
    print(f"{iteration_path} already exists. Exiting to preserve previous work.")
    sys.exit(0)
except OSError:
    print("An error occurred while creating the folder. ")



c:\Users\timef\Documents\Workspaces\Python\csc180\bird-song-recognizer\output already exists


## Import and Read Datasets

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
train_mp3_base = os.path.join(mp3_dataset_base_path, 'train')
test_mp3_base = os.path.join(mp3_dataset_base_path, 'test')
val_mp3_base = os.path.join(mp3_dataset_base_path, 'val')
# grab all mp3 files in a directory
train_mp3_file_list = get_all_files(train_mp3_base)
test_mp3_file_list = get_all_files(test_mp3_base)
val_mp3_file_list = get_all_files(val_mp3_base)
print(f"train mp3 base: {train_mp3_base}\n test mp3 base: {test_mp3_base}\nval mp3 base: {val_mp3_base}\n lists:{train_mp3_file_list}\n{test_mp3_file_list}\n{val_mp3_file_list}")
for file, i in enumerate(train_mp3_file_list):
    if i > 1:
        break
    data, sample_rate = librosa.load(file)
    mel_spectrogram = librosa.feature.melspectrogram(y=data, sr=sample_rate)
    # Convert to Decibels (Log Scale)
    # Convert to decibels (log scale): Spectrograms are often displayed in decibels for better visualization of dynamic ranges.
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spectrogram_db, x_axis='time', y_axis='mel', sr=sample_rate, cmap='viridis')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel Spectrogram')
    plt.savefig(os.path.join(spectrogram_train_path, f"{i}.png"))
    plt.close()





hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit
hit


: 